In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df = pd.read_excel("/Users/senarajayawardena/Documents/Dialog/20260512_FintechTrainee_CaseStudy_Dataset_FinSightLanka_F.xlsx")


Data Cleaning

In [3]:
# Remove duplicates
dup_mask = df["Customer_ID"].duplicated() | df["NIC_Number"].duplicated()
df = df[dup_mask].reset_index(drop=True)
print(f"Removed {dup_mask.sum()} duplicates → {len(df):,} rows remain")

Removed 11 duplicates → 11 rows remain


In [6]:
# Gender
df["Gender"] = df["Gender"].str.strip().str.capitalize().replace({"M": "Male", "F": "Female"})

# Account Status
df["Account_Status"] = df["Account_Status"].str.strip().str.capitalize()

# District typos
df["District"] = df["District"].replace({"Columbo": "Colombo", "Kanady": "Kandy", "Gale": "Galle", "Jafna": "Jaffna"})

# Province from District
DISTRICT_TO_PROVINCE = {
    "Colombo": "Western", "Gampaha": "Western", "Kandy": "Central",
    "Badulla": "Uva", "Trincomalee": "Eastern", "Jaffna": "Northern",
    "Galle": "Southern", "Matara": "Southern", "Kurunegala": "North Western",
    "Ratnapura": "Sabaragamuwa",
}
df["Province"] = df["District"].map(DISTRICT_TO_PROVINCE).combine_first(df["Province"])

In [ ]:
# counts missing values per column and shows only columns that have at least one missing value,
# sorted by most missing first.

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
print(pd.DataFrame({"Count": missing, "%": missing_pct})
      .query("Count > 0")
      .sort_values("Count", ascending=False))

                          Count     %
FD_Amount                   358  70.5
Loan_Amount                 291  57.3
Loan_Tenure_Months          290  57.1
Monthly_Repayment_Amount    290  57.1
Loan_Type                   290  57.1
Outstanding_Loan_Balance    289  56.9
Loan_Start_Date             288  56.7
Loan_Repayment_Status       288  56.7
Referral_Source             252  49.6
Last_Login_Date             169  33.3
Full_Name                     8   1.6
KYC_Status                    7   1.4
Monthly_Deposit_Avg           6   1.2
Savings_Balance               6   1.2
District                      5   1.0
Has_Loan                      5   1.0
Customer_Segment              5   1.0


In [8]:
# Null out bad values
df.loc[df["Savings_Balance"] < 0, "Savings_Balance"] = np.nan
df.loc[df["Savings_Balance"] >= 999_000_000, "Savings_Balance"] = np.nan

# Impute with segment median
df["Savings_Balance"] = df.groupby("Customer_Segment")["Savings_Balance"] \
                          .transform(lambda x: x.fillna(x.median()))

In [9]:
# nulls out negative deposits (impossible), 
# fills missing deposits and withdrawals with their median, 
# then recalculates net flow.

# Fix deposits
df.loc[df["Monthly_Deposit_Avg"] < 0, "Monthly_Deposit_Avg"] = np.nan
df["Monthly_Deposit_Avg"] = df["Monthly_Deposit_Avg"].fillna(df["Monthly_Deposit_Avg"].median())

# Fix withdrawals
df["Monthly_Withdrawal_Avg"] = df["Monthly_Withdrawal_Avg"].fillna(df["Monthly_Withdrawal_Avg"].median())

# Recompute net flow
df["Net_Monthly_Flow"] = df["Monthly_Deposit_Avg"] - df["Monthly_Withdrawal_Avg"]

In [11]:
#fills missing Has_Loan based on whether Loan_Amount exists, 
#then checks that non-borrowers don't have any loan data filled in.


# Infer Has_Loan from Loan_Amount
mask = df["Has_Loan"].isna()
df.loc[mask & df["Loan_Amount"].notna(), "Has_Loan"] = "Yes"
df.loc[mask & df["Loan_Amount"].isna(),  "Has_Loan"] = "No"

# Check for spurious loan data on non-borrowers
loan_cols = ["Loan_Amount", "Loan_Tenure_Months", "Monthly_Repayment_Amount",
             "Outstanding_Loan_Balance", "Loan_Type", "Loan_Repayment_Status"]

spurious = df.loc[df["Has_Loan"] == "No", loan_cols].notna().sum()
print(spurious[spurious > 0])  # prints only problem columns

Loan_Amount                 1
Loan_Tenure_Months          1
Monthly_Repayment_Amount    1
Outstanding_Loan_Balance    2
Loan_Type                   1
Loan_Repayment_Status       3
dtype: int64


In [12]:
for col in ["Customer_Segment", "District", "KYC_Status"]:
    df[col] = df[col].fillna("Unknown")

In [14]:
#recalculates tenure from Account_Open_Date and overwrites the existing column
#(ignoring whatever was there before).

REF_DATE = pd.Timestamp("2026-05-01")
df["Customer_Tenure_Years"] = ((REF_DATE - df["Account_Open_Date"]).dt.days / 365.25).round(1)



In [15]:
#creates 5 new columns 
#age groups, debt-to-savings ratio, savings growth flag, cross-sell candidates, and high-risk flag.


# Age bands
df["Age_Band"] = pd.cut(df["Age"], bins=[17,24,34,44,54,64,100],
                        labels=["18-24","25-34","35-44","45-54","55-64","65+"])

# Debt-to-Savings (loan holders with positive savings only)
mask = (df["Has_Loan"] == "Yes") & df["Outstanding_Loan_Balance"].notna() & (df["Savings_Balance"] > 0)
df["Debt_to_Savings"] = np.where(mask, df["Outstanding_Loan_Balance"] / df["Savings_Balance"], np.nan)

# Savings growing
df["Savings_Growing"] = df["Net_Monthly_Flow"] > 0

# Cross-sell opportunity
df["CrossSell_Opportunity"] = (df["Savings_Balance"] > 0) & (df["Has_Fixed_Deposit"] == "No") & (df["Has_Insurance"] == "No")

# High risk
df["High_Risk_Flag"] = (df["Debt_to_Savings"] > 1.0) & (df["Loan_Repayment_Status"] == "Defaulted")
